# **Installation and Licensing**

In [ ]:
!pip install -q gamspy # Installs the GAMSPy package. GAMSPy documentation: https://gamspy.readthedocs.io/en/latest/

In [ ]:
!gamspy show license

In [ ]:
!gamspy list solvers

In [ ]:
!gamspy install license <path_to_ascii_file or access code> # paste your license here (if demo-license is not enough)

In [ ]:
# Optional: Install all solvers
!gamspy install solver --install-all-solvers

In [ ]:
!gamspy list solvers # Lists all available solvers in the gamspy package

# **Mathematical Model**

**Multi-Period Crypto Portfolio Optimization Problem**

\begin{align}
    \min \quad & \lambda \cdot \frac{\sum_{t} \sum_{i,j} Q_{t,ij} x_{t,i} x_{t,j}}{\sum |Q_{t,ij}| + \epsilon}
    - (1-\lambda) \cdot \frac{\sum_{t} \sum_i \text{ER}_{t,i} \cdot x_{t,i}}{\sum |\text{ER}_{t,i}| + \epsilon} \\
    \text{s.t.} \quad
    & \sum_{i=1}^n x_{t,i} = 1 \quad && \forall t \in T \quad \text{(Budget)} \\
    & \sum_{i=1}^n y_{t,i} \leq 12 \quad && \forall t \in T \quad \text{(Cardinality)} \\
    & x_{t,i} \leq y_{t,i} \quad && \forall t, i \quad \text{(Linking)} \\
    & x_{t,i} \geq 0.01 \cdot y_{t,i} \quad && \forall t, i \quad \text{(Min investment)} \\
    & \delta_{t,i} = x_{t,i} - x_{t-1,i} \quad && \forall t > 1,\ i \quad \text{(Trade dynamics)} \\
    & x_{t,i} \geq 0,\ y_{t,i} \in \{0,1\},\ \delta_{t,i} \in \mathbb{R} \quad && \forall t,\ i \quad \text{(Domains)}
\end{align}

where $λ=0.4$.

$$
\begin{array}{l l l r r r r}
\text{Name} & \text{Type} & \text{C} & \#\text{Vars} & \#\text{BinVars} & \#\text{IntVars} & \#\text{Cons} \\
\hline
MPCCPOP\_crypto8\_1mo\_1d & MBQP & Convex & 481 & 240 & 0 & 541 \\
MPCCPOP\_crypto8\_1mo\_5d & MBQP & Convex & 97 & 48 & 0 & 109\\
MPCCPOP\_crypto8\_6d\_30m & MBQP & Convex & 4081 & 2040 & 0 & 4591 \\
MPCCPOP\_crypto8\_30d\_90m & MBQP & Convex & 7505 & 3752 & 0 & 8443 
\end{array}
$$


#  **Solver Setup**


## MPCCPOP\_crypto8\_1mo\_1d


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
from gamspy.math import abs


# --- Download hourly crypto data ---
tickers = ["BTC-USD", "ETH-USD", "BNB-USD", "XRP-USD", "SOL-USD", "TRX-USD", "DOGE-USD", "WTRX-USD"] # , "USDT-USD", "USDC-USD",
data = yf.download(tickers, period="1mo", interval="1d")['Close']

returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = len(returns)
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]


for lambda_ in [0.4]:

# Initialize GAMSPy Container and Sets
    m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
    i = Set(m, name="i", records=assets)
    ii = Alias(m, name="ii", alias_with=i)

    t = Set(m, name="t", records=time_periods)
    tt = Alias(m, name="tt", alias_with=t)

# Parameters
    Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
    q_records = []
    for t_idx, tp in enumerate(time_periods):
        cov = cov_list[t_idx]
        q_records += [(tp, assets[r], assets[c], cov[r, c])
                      for r in range(len(assets)) for c in range(len(assets))]

    # Convert to DataFrame with categorical columns
    df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
    df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
    df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
    df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
    Q_t.records = df_q


# 1. Calculate expected returns (already have returns DataFrame)
    expected_returns = returns

# 2. Build expected returns parameter ER_t
    er_records = []
    for t_idx, tp in enumerate(time_periods):
        for asset_idx, asset in enumerate(assets):
            er_value = expected_returns.iloc[t_idx, asset_idx]
            er_records.append((tp, asset, er_value))

    df_er = pd.DataFrame(er_records, columns=["t", "i", "value"])
    df_er["t"] = pd.Categorical(df_er["t"], categories=time_periods)
    df_er["i"] = pd.Categorical(df_er["i"], categories=assets)

    ER_t = Parameter(m, name="ER_t", domain=[t, i])
    ER_t.records = df_er

# Variables
    x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
    y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
    delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
    budget = Equation(m, name="budget", domain=[t])
    budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
    cardinality = Equation(m, name="cardinality", domain=[t])
    cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
    if len(time_periods) > 1:
        link_trade = Equation(m, name="link_trade", domain=[t, i])
        for t_idx in range(1, len(time_periods)):
            curr_t = time_periods[t_idx]
            prev_t = time_periods[t_idx - 1]
            for asset in assets:
                link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

    # Link x and y (e.g. enforce x[t, i] <= y[t, i])
    link_x_y = Equation(m, name="link_x_y", domain=[t, i])
    link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
    invest = Equation(m, name="invest", domain=[t, i])
    invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Use sum of absolute values directly from DataFrame-style records
    ER_scale = ER_t.records["value"].abs().sum()
    Q_scale = Q_t.records["value"].abs().sum()

# Raw terms
    expected_return_raw = Sum([t, i], ER_t[t, i] * x[t, i])
    portfolio_variance_raw = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])

# Normalize (add small epsilon to avoid division by zero)
    epsilon = 1e-6
    expected_return_term = expected_return_raw / (ER_scale + epsilon)
    portfolio_variance = portfolio_variance_raw / (Q_scale + epsilon)

    import sys

# Trade-off parameter
    print(f"Running for lambda = {lambda_}")

    obj = lambda_ * portfolio_variance - (1 - lambda_) * expected_return_term

    model_name = f"MultiPeriodPortfolio_lambda_{str(lambda_).replace('.', 'p')}"
    model = Model(m, name="MPCCPOP_crypto8_1mo_1d",
                  equations=m.getEquations(),
                  problem="MIQCP", sense=Sense.MIN, objective=obj)

    model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 0}) #, "Model.Convexity.AssumeConvex": True

In [ ]:
# Convert GAMSPy model to GAMS (Note: One needs to download both the .gms and .gdx file)
model.toGams(path="gams")

After this, the files were opened and combined in GAMS Studio by GAMS Convert. Below is the rest of the codes for this type of problem.


## MPCCPOP\_crypto8\_1mo\_5d


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
from gamspy.math import abs


# --- Download hourly crypto data ---
tickers = ["BTC-USD", "ETH-USD", "BNB-USD", "XRP-USD", "SOL-USD", "TRX-USD", "DOGE-USD", "WTRX-USD"] # , "USDT-USD", "USDC-USD",
data = yf.download(tickers, period="1mo", interval="5d")['Close']

returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = len(returns)
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]


for lambda_ in [0.4]:

# Initialize GAMSPy Container and Sets
    m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
    i = Set(m, name="i", records=assets)
    ii = Alias(m, name="ii", alias_with=i)

    t = Set(m, name="t", records=time_periods)
    tt = Alias(m, name="tt", alias_with=t)

# Parameters
    Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
    q_records = []
    for t_idx, tp in enumerate(time_periods):
        cov = cov_list[t_idx]
        q_records += [(tp, assets[r], assets[c], cov[r, c])
                      for r in range(len(assets)) for c in range(len(assets))]

    # Convert to DataFrame with categorical columns
    df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
    df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
    df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
    df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
    Q_t.records = df_q


# 1. Calculate expected returns (already have returns DataFrame)
    expected_returns = returns

# 2. Build expected returns parameter ER_t
    er_records = []
    for t_idx, tp in enumerate(time_periods):
        for asset_idx, asset in enumerate(assets):
            er_value = expected_returns.iloc[t_idx, asset_idx]
            er_records.append((tp, asset, er_value))

    df_er = pd.DataFrame(er_records, columns=["t", "i", "value"])
    df_er["t"] = pd.Categorical(df_er["t"], categories=time_periods)
    df_er["i"] = pd.Categorical(df_er["i"], categories=assets)

    ER_t = Parameter(m, name="ER_t", domain=[t, i])
    ER_t.records = df_er

# Variables
    x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
    y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
    delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
    budget = Equation(m, name="budget", domain=[t])
    budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
    cardinality = Equation(m, name="cardinality", domain=[t])
    cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
    if len(time_periods) > 1:
        link_trade = Equation(m, name="link_trade", domain=[t, i])
        for t_idx in range(1, len(time_periods)):
            curr_t = time_periods[t_idx]
            prev_t = time_periods[t_idx - 1]
            for asset in assets:
                link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

    # Link x and y (e.g. enforce x[t, i] <= y[t, i])
    link_x_y = Equation(m, name="link_x_y", domain=[t, i])
    link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
    invest = Equation(m, name="invest", domain=[t, i])
    invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Use sum of absolute values directly from DataFrame-style records
    ER_scale = ER_t.records["value"].abs().sum()
    Q_scale = Q_t.records["value"].abs().sum()

# Raw terms
    expected_return_raw = Sum([t, i], ER_t[t, i] * x[t, i])
    portfolio_variance_raw = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])

# Normalize (add small epsilon to avoid division by zero)
    epsilon = 1e-6
    expected_return_term = expected_return_raw / (ER_scale + epsilon)
    portfolio_variance = portfolio_variance_raw / (Q_scale + epsilon)

    import sys

# Trade-off parameter
    print(f"Running for lambda = {lambda_}")

    obj = lambda_ * portfolio_variance - (1 - lambda_) * expected_return_term

    model_name = f"MultiPeriodPortfolio_lambda_{str(lambda_).replace('.', 'p')}"
    model = Model(m, name="MPCCPOP_crypto8_1mo_5d",
                  equations=m.getEquations(),
                  problem="MIQCP", sense=Sense.MIN, objective=obj)

    model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 0}) #, "Model.Convexity.AssumeConvex": True


## MPCCPOP\_crypto8\_6d\_30m

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
from gamspy.math import abs


# --- Download hourly crypto data ---
tickers = ["BTC-USD", "ETH-USD", "BNB-USD", "XRP-USD", "SOL-USD", "TRX-USD", "DOGE-USD", "WTRX-USD"] # , "USDT-USD", "USDC-USD",
data = yf.download(tickers, period="6d", interval="30m")['Close']

returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = len(returns)
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]


for lambda_ in [0.4]:

# Initialize GAMSPy Container and Sets
    m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
    i = Set(m, name="i", records=assets)
    ii = Alias(m, name="ii", alias_with=i)

    t = Set(m, name="t", records=time_periods)
    tt = Alias(m, name="tt", alias_with=t)

# Parameters
    Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
    q_records = []
    for t_idx, tp in enumerate(time_periods):
        cov = cov_list[t_idx]
        q_records += [(tp, assets[r], assets[c], cov[r, c])
                      for r in range(len(assets)) for c in range(len(assets))]

    # Convert to DataFrame with categorical columns
    df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
    df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
    df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
    df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
    Q_t.records = df_q


# 1. Calculate expected returns (already have returns DataFrame)
    expected_returns = returns

# 2. Build expected returns parameter ER_t
    er_records = []
    for t_idx, tp in enumerate(time_periods):
        for asset_idx, asset in enumerate(assets):
            er_value = expected_returns.iloc[t_idx, asset_idx]
            er_records.append((tp, asset, er_value))

    df_er = pd.DataFrame(er_records, columns=["t", "i", "value"])
    df_er["t"] = pd.Categorical(df_er["t"], categories=time_periods)
    df_er["i"] = pd.Categorical(df_er["i"], categories=assets)

    ER_t = Parameter(m, name="ER_t", domain=[t, i])
    ER_t.records = df_er

# Variables
    x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
    y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
    delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
    budget = Equation(m, name="budget", domain=[t])
    budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
    cardinality = Equation(m, name="cardinality", domain=[t])
    cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
    if len(time_periods) > 1:
        link_trade = Equation(m, name="link_trade", domain=[t, i])
        for t_idx in range(1, len(time_periods)):
            curr_t = time_periods[t_idx]
            prev_t = time_periods[t_idx - 1]
            for asset in assets:
                link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

    # Link x and y (e.g. enforce x[t, i] <= y[t, i])
    link_x_y = Equation(m, name="link_x_y", domain=[t, i])
    link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
    invest = Equation(m, name="invest", domain=[t, i])
    invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Use sum of absolute values directly from DataFrame-style records
    ER_scale = ER_t.records["value"].abs().sum()
    Q_scale = Q_t.records["value"].abs().sum()

# Raw terms
    expected_return_raw = Sum([t, i], ER_t[t, i] * x[t, i])
    portfolio_variance_raw = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])

# Normalize (add small epsilon to avoid division by zero)
    epsilon = 1e-6
    expected_return_term = expected_return_raw / (ER_scale + epsilon)
    portfolio_variance = portfolio_variance_raw / (Q_scale + epsilon)

    import sys

# Trade-off parameter
    print(f"Running for lambda = {lambda_}")

    obj = lambda_ * portfolio_variance - (1 - lambda_) * expected_return_term

    model_name = f"MultiPeriodPortfolio_lambda_{str(lambda_).replace('.', 'p')}"
    model = Model(m, name="MPCCPOP_crypto8_6d_30m",
                  equations=m.getEquations(),
                  problem="MIQCP", sense=Sense.MIN, objective=obj)

    #model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2}) #, "Model.Convexity.AssumeConvex": True

    model.solve(output=sys.stdout, solver = "CONVERT")

## MPCCPOP\_crypto8\_30d\_90m

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias
from gamspy.math import abs


# --- Download hourly crypto data ---
tickers = ["BTC-USD", "ETH-USD", "BNB-USD", "XRP-USD", "SOL-USD", "TRX-USD", "DOGE-USD", "WTRX-USD"] # , "USDT-USD", "USDC-USD",
data = yf.download(tickers, period="30d", interval="90m")['Close']

returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = len(returns)
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]


for lambda_ in [0.4]:

# Initialize GAMSPy Container and Sets
    m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
    i = Set(m, name="i", records=assets)
    ii = Alias(m, name="ii", alias_with=i)

    t = Set(m, name="t", records=time_periods)
    tt = Alias(m, name="tt", alias_with=t)

# Parameters
    Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
    q_records = []
    for t_idx, tp in enumerate(time_periods):
        cov = cov_list[t_idx]
        q_records += [(tp, assets[r], assets[c], cov[r, c])
                      for r in range(len(assets)) for c in range(len(assets))]

    # Convert to DataFrame with categorical columns
    df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
    df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
    df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
    df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
    Q_t.records = df_q


# 1. Calculate expected returns (already have returns DataFrame)
    expected_returns = returns

# 2. Build expected returns parameter ER_t
    er_records = []
    for t_idx, tp in enumerate(time_periods):
        for asset_idx, asset in enumerate(assets):
            er_value = expected_returns.iloc[t_idx, asset_idx]
            er_records.append((tp, asset, er_value))

    df_er = pd.DataFrame(er_records, columns=["t", "i", "value"])
    df_er["t"] = pd.Categorical(df_er["t"], categories=time_periods)
    df_er["i"] = pd.Categorical(df_er["i"], categories=assets)

    ER_t = Parameter(m, name="ER_t", domain=[t, i])
    ER_t.records = df_er

# Variables
    x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
    y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
    delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
    budget = Equation(m, name="budget", domain=[t])
    budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
    cardinality = Equation(m, name="cardinality", domain=[t])
    cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
    if len(time_periods) > 1:
        link_trade = Equation(m, name="link_trade", domain=[t, i])
        for t_idx in range(1, len(time_periods)):
            curr_t = time_periods[t_idx]
            prev_t = time_periods[t_idx - 1]
            for asset in assets:
                link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

    # Link x and y (e.g. enforce x[t, i] <= y[t, i])
    link_x_y = Equation(m, name="link_x_y", domain=[t, i])
    link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
    invest = Equation(m, name="invest", domain=[t, i])
    invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Use sum of absolute values directly from DataFrame-style records
    ER_scale = ER_t.records["value"].abs().sum()
    Q_scale = Q_t.records["value"].abs().sum()

# Raw terms
    expected_return_raw = Sum([t, i], ER_t[t, i] * x[t, i])
    portfolio_variance_raw = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])

# Normalize (add small epsilon to avoid division by zero)
    epsilon = 1e-6
    expected_return_term = expected_return_raw / (ER_scale + epsilon)
    portfolio_variance = portfolio_variance_raw / (Q_scale + epsilon)

    import sys

# Trade-off parameter
    print(f"Running for lambda = {lambda_}")

    obj = lambda_ * portfolio_variance - (1 - lambda_) * expected_return_term

    model_name = f"MultiPeriodPortfolio_lambda_{str(lambda_).replace('.', 'p')}"
    model = Model(m, name="MPCCPOP_crypto8_30d_90m",
                  equations=m.getEquations(),
                  problem="MIQCP", sense=Sense.MIN, objective=obj)

    model.solve(output=sys.stdout, solver = "CONVERT")